# 06 - Dynamic Regime-Aware Head Selection with 75% Keep

Bu notebook, **B4 PatchTST baseline** üzerinde daha adil bir dynamic regime-aware selection deneyi çalıştırır.

Bu kez dynamic yöntem:

```text
24 head içinden 18 head açık
6 head kapalı
```

kullanır. Yani static pruning 25% ile aynı pruning oranında karşılaştırılabilir.

Amaç:

1. B4 checkpoint'ini yüklemek  
2. Regime etiketlerini okumak  
3. Head importance tablosundan her regime için en iyi 18 head'i seçmek  
4. Validation sırasında her pencereye kendi regime'ine göre maske uygulamak  
5. Baseline, static 25%, dynamic 50% ve dynamic 75% sonuçlarını karşılaştırmak  
6. Sonuçları Drive'a kaydetmek  

Bu notebook test değil validation odaklıdır. Dynamic test için test pencerelerine de STL regime label üretmek gerekir.


## 1. Drive bağla ve importlar

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys
import os
import shutil
import random
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from tqdm.auto import tqdm

## 2. Proje yolları

In [3]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BIL401_Regime_Head_Pruning"
)

REGIME_DIR = PROJECT_DIR / "regime_detection"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
HEAD_IMPORTANCE_DIR = PROJECT_DIR / "head_importance"
PRUNING_DIR = PROJECT_DIR / "pruning_experiments"

STATIC_DIR = PRUNING_DIR / "b4_static_pruning_25"
DYNAMIC_50_DIR = PRUNING_DIR / "b4_dynamic_regime_aware_50_keep"
DYNAMIC_75_DIR = PRUNING_DIR / "b4_dynamic_regime_aware_75_keep"

DYNAMIC_75_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("REGIME_DIR:", REGIME_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("HEAD_IMPORTANCE_DIR:", HEAD_IMPORTANCE_DIR)
print("PRUNING_DIR:", PRUNING_DIR)
print("STATIC_DIR:", STATIC_DIR)
print("DYNAMIC_50_DIR:", DYNAMIC_50_DIR)
print("DYNAMIC_75_DIR:", DYNAMIC_75_DIR)

PROJECT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning
REGIME_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/regime_detection
CHECKPOINT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints
HEAD_IMPORTANCE_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/head_importance
PRUNING_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments
STATIC_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_static_pruning_25
DYNAMIC_50_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_dynamic_regime_aware_50_keep
DYNAMIC_75_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_dynamic_regime_aware_75_keep


## 3. Time-Series-Library ve dependency hazırlığı

In [4]:
TSLIB_DIR = Path("/content/Time-Series-Library")

if not TSLIB_DIR.exists():
    %cd /content
    !git clone https://github.com/thuml/Time-Series-Library.git
else:
    print("Time-Series-Library already exists:", TSLIB_DIR)

sys.path.insert(0, str(TSLIB_DIR))
print("Python path[0]:", sys.path[0])

/content
Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 2295, done.
remote: Total 2295 (delta 0), reused 0 (delta 0), pack-reused 2295 (from 1)
Receiving objects: 100% (2295/2295), 78.43 MiB | 28.42 MiB/s, done.
Resolving deltas: 100% (1570/1570), done.
Python path[0]: /content/Time-Series-Library


In [5]:
# TSLib importları için minimal paketler
!pip install -q patool sktime scikit-base --no-deps

# Bazı Time-Series-Library model importları için gerekli olabiliyor.
!pip install -q reformer-pytorch --no-deps
!pip install -q local-attention --no-deps
!pip install -q hyper_connections --no-deps
!pip install -q axial_positional_embedding --no-deps
!pip install -q product_key_memory --no-deps
!pip install -q colt5_attention --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 13.6 MB/s eta 0:00:00


In [6]:
drive_data_path = PROJECT_DIR / "data" / "ETTh1.csv"

tslib_data_path = (
    TSLIB_DIR
    / "dataset/ETDataset/ETT-small/ETTh1.csv"
)

tslib_data_path.parent.mkdir(parents=True, exist_ok=True)

if drive_data_path.exists():
    shutil.copy2(drive_data_path, tslib_data_path)
    print("ETTh1 copied from Drive.")
else:
    print("Drive data not found. Downloading ETTh1...")
    !wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv

print("Dataset exists:", tslib_data_path.exists())
print("Dataset path:", tslib_data_path)

df = pd.read_csv(tslib_data_path)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Drive data not found. Downloading ETTh1...
Dataset exists: True
Dataset path: /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv
Dataset shape: (17420, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


## 4. Regime ve head importance dosyalarını oku

In [7]:
regime_path = REGIME_DIR / "etth1_validation_regimes_ot_seq336.csv"
regime_df = pd.read_csv(regime_path)

print("Regime df shape:", regime_df.shape)
display(regime_df.head())
display(regime_df["regime"].value_counts())

Regime df shape: (2785, 13)


,window_id,start_idx,end_idx,input_start_date,input_end_date,trend_score,seasonal_score,residual_score,regime,top_score,second_score,confidence_margin,is_confident
0,0,0,336,2017-06-12 00:00:00,2017-06-25 23:00:00,0.700476,0.205640,0.093884,trend,0.700476,0.205640,0.494836,True
1,1,1,337,2017-06-12 01:00:00,2017-06-26 00:00:00,0.700166,0.206039,0.093796,trend,0.700166,0.206039,0.494127,True
2,2,2,338,2017-06-12 02:00:00,2017-06-26 01:00:00,0.696768,0.208689,0.094543,trend,0.696768,0.208689,0.488079,True
3,3,3,339,2017-06-12 03:00:00,2017-06-26 02:00:00,0.689886,0.213555,0.096559,trend,0.689886,0.213555,0.476330,True
4,4,4,340,2017-06-12 04:00:00,2017-06-26 03:00:00,0.677356,0.223502,0.099142,trend,0.677356,0.223502,0.453854,True


,count
regime,
trend,2134
residual,359
seasonal,292


In [8]:
importance_path = HEAD_IMPORTANCE_DIR / "b4_head_importance_all_windows.csv"
importance_df = pd.read_csv(importance_path)

print("Importance df shape:", importance_df.shape)
display(importance_df.head())

assert len(importance_df) == 24, "B4 için 24 head bekleniyor."

Importance df shape: (24, 14)


,layer,head,baseline_overall_mse,masked_overall_mse,overall_importance,baseline_trend_mse,masked_trend_mse,trend_importance,baseline_seasonal_mse,masked_seasonal_mse,seasonal_importance,baseline_residual_mse,masked_residual_mse,residual_importance
0,0,0,0.678066,0.682159,0.004092,0.67695,0.680246,0.003296,0.706689,0.708879,0.002190,0.661419,0.671793,0.010374
1,0,1,0.678066,0.695225,0.017158,0.67695,0.697562,0.020611,0.706689,0.695852,-0.010837,0.661419,0.680825,0.019406
2,0,2,0.678066,0.677773,-0.000293,0.67695,0.673978,-0.002973,0.706689,0.685340,-0.021349,0.661419,0.694178,0.032759
3,0,3,0.678066,0.672993,-0.005073,0.67695,0.679229,0.002279,0.706689,0.632469,-0.074220,0.661419,0.668884,0.007465
4,0,4,0.678066,0.687903,0.009836,0.67695,0.699850,0.022900,0.706689,0.646941,-0.059748,0.661419,0.650203,-0.011217


## 5. Dynamic 75% keep head listelerini oluştur

Önceki dynamic 50% notebookunda her regime için 12 head açık bırakmıştık.  
Burada her regime için **18 head** açık bırakıyoruz.

Bu, static pruning 25% ile aynı aktif head sayısıdır:

```text
Static 25%: 18 active / 6 pruned
Dynamic 75% keep: 18 active / 6 pruned
```


In [9]:
def get_top_heads_for_regime(
    importance_df,
    regime,
    keep_ratio=0.75,
):
    importance_col = f"{regime}_importance"

    num_total = len(importance_df)
    num_keep = int(num_total * keep_ratio)

    top_heads = (
        importance_df
        .sort_values(importance_col, ascending=False)
        .head(num_keep)
        .copy()
        .reset_index(drop=True)
    )

    return top_heads


KEEP_RATIO = 0.75

dynamic_keep_dfs = {
    "trend": get_top_heads_for_regime(
        importance_df,
        "trend",
        keep_ratio=KEEP_RATIO,
    ),
    "seasonal": get_top_heads_for_regime(
        importance_df,
        "seasonal",
        keep_ratio=KEEP_RATIO,
    ),
    "residual": get_top_heads_for_regime(
        importance_df,
        "residual",
        keep_ratio=KEEP_RATIO,
    ),
}

for regime, keep_df in dynamic_keep_dfs.items():
    print(f"\n{regime} keep heads:", len(keep_df))
    display(keep_df[["layer", "head", f"{regime}_importance"]])
    assert len(keep_df) == 18, f"{regime} için 18 head açık bekleniyor."



trend keep heads: 18


,layer,head,trend_importance
0,0,4,0.022900
1,0,1,0.020611
2,2,2,0.012905
3,2,6,0.011054
4,1,6,0.010993
5,1,0,0.008527
6,0,7,0.005714
7,1,4,0.005648
8,1,5,0.003802
9,0,0,0.003296



seasonal keep heads: 18


,layer,head,seasonal_importance
0,2,1,0.036730
1,2,3,0.022212
2,2,7,0.009473
3,1,0,0.006389
4,0,0,0.002190
5,2,2,-0.005710
6,0,6,-0.009039
7,0,1,-0.010837
8,0,2,-0.021349
9,0,7,-0.023027



residual keep heads: 18


,layer,head,residual_importance
0,1,3,0.087188
1,1,6,0.056504
2,2,7,0.055780
3,2,1,0.046922
4,1,1,0.042267
5,2,3,0.040165
6,0,5,0.039628
7,1,0,0.039080
8,0,7,0.035285
9,0,6,0.033360


In [10]:
# Keep listelerini hemen kaydet
for regime, keep_df in dynamic_keep_dfs.items():
    keep_df.to_csv(
        DYNAMIC_75_DIR / f"{regime}_dynamic_keep_75_heads.csv",
        index=False,
    )

print("Dynamic 75% keep head lists saved to:")
print(DYNAMIC_75_DIR)

Dynamic 75% keep head lists saved to:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_dynamic_regime_aware_75_keep


## 6. B4 checkpoint yolunu bul

In [11]:
b4_checkpoint_candidates = list(
    CHECKPOINT_DIR.glob(
        "B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth"
    )
)

print("Found checkpoint candidates:")
for path in b4_checkpoint_candidates:
    print(path)

if len(b4_checkpoint_candidates) == 0:
    raise FileNotFoundError(
        "B4 checkpoint bulunamadı. CHECKPOINT_DIR içini kontrol et."
    )

b4_checkpoint_path = b4_checkpoint_candidates[0]
print("\nSelected checkpoint:")
print(b4_checkpoint_path)

Found checkpoint candidates:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth

Selected checkpoint:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth


## 7. B4 model argümanları

In [12]:
from argparse import Namespace

args = Namespace(
    # task
    task_name="long_term_forecast",
    is_training=0,
    model_id="ETTh1_336_96_dm128_h8",
    model="PatchTST",

    # data
    data="ETTh1",
    root_path="./dataset/ETDataset/ETT-small/",
    data_path="ETTh1.csv",
    features="M",
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",

    # forecasting
    seq_len=336,
    label_len=48,
    pred_len=96,
    seasonal_patterns="Monthly",
    inverse=False,

    # model
    enc_in=7,
    dec_in=7,
    c_out=7,
    d_model=128,
    n_heads=8,
    e_layers=3,
    d_layers=1,
    d_ff=256,
    moving_avg=25,
    factor=3,
    distil=True,
    dropout=0.1,
    embed="timeF",
    activation="gelu",
    output_attention=False,

    # PatchTST related
    patch_len=16,
    stride=8,
    padding_patch="end",
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,

    # optimization / loader
    num_workers=0,
    itr=1,
    train_epochs=10,
    batch_size=32,
    patience=3,
    learning_rate=0.0001,
    des="baseline_b4",
    loss="MSE",
    lradj="type1",
    use_amp=False,

    # GPU
    use_gpu=torch.cuda.is_available(),
    gpu_type="cuda",
    gpu=0,
    use_multi_gpu=False,
    devices="0",

    # other model families, required by some imports
    expand=2,
    d_conv=4,
    top_k=5,
    num_kernels=6,
    channel_independence=0,
    decomp_method="moving_avg",
    use_norm=1,
    down_sampling_layers=0,
    down_sampling_window=1,
    down_sampling_method=None,
    seg_len=48,

    # MLP projection args sometimes expected
    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
)

print(args)

Namespace(task_name='long_term_forecast', is_training=0, model_id='ETTh1_336_96_dm128_h8', model='PatchTST', data='ETTh1', root_path='./dataset/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=336, label_len=48, pred_len=96, seasonal_patterns='Monthly', inverse=False, enc_in=7, dec_in=7, c_out=7, d_model=128, n_heads=8, e_layers=3, d_layers=1, d_ff=256, moving_avg=25, factor=3, distil=True, dropout=0.1, embed='timeF', activation='gelu', output_attention=False, patch_len=16, stride=8, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, num_workers=0, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='baseline_b4', loss='MSE', lradj='type1', use_amp=False, use_gpu=True, gpu_type='cuda', gpu=0, use_multi_gpu=False, devices='0', expand=2, d_conv=4, top_k=5, num_kernels=6, channel_independence=0, decomp_method='moving_avg', use_norm=1, down_s

## 8. Modeli yükle

In [13]:
%cd /content/Time-Series-Library

/content/Time-Series-Library


In [14]:
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

exp = Exp_Long_Term_Forecast(args)
model = exp.model.to(device)

checkpoint = torch.load(
    b4_checkpoint_path,
    map_location=device,
)

model.load_state_dict(checkpoint)
model.eval()

print("Model loaded successfully.")

Device: cuda:0
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
Model loaded successfully.


## 9. Validation loader

In [15]:
from data_provider.data_factory import data_provider

vali_data, vali_loader = data_provider(
    args,
    flag="val",
)

print("Validation dataset length:", len(vali_data))
print("Validation loader batches:", len(vali_loader))

assert len(vali_data) == len(regime_df), (
    len(vali_data),
    len(regime_df),
)

print("Validation windows and regime labels match.")

val 2785
Validation dataset length: 2785
Validation loader batches: 88
Validation windows and regime labels match.


## 10. DynamicHeadMaskController

PatchTST channel-independent çalıştığı için attention içine giren batch boyutu:

```text
original_batch × channel_count
```

olabiliyor. Bu yüzden batch-level maskeyi encoder batch boyutuna genişletiyoruz.


In [16]:
class DynamicHeadMaskController:
    def __init__(self, model, num_channels=7):
        self.model = model
        self.num_channels = num_channels
        self.original_forwards = {}
        self.current_mask = None

    def install(self):
        for layer_idx, encoder_layer in enumerate(self.model.encoder.attn_layers):
            attention_layer = encoder_layer.attention

            if layer_idx in self.original_forwards:
                continue

            original_forward = attention_layer.forward
            self.original_forwards[layer_idx] = original_forward

            def make_masked_forward(layer_idx, attention_layer):
                def masked_forward(
                    queries,
                    keys,
                    values,
                    attn_mask,
                    tau=None,
                    delta=None,
                ):
                    B, L, _ = queries.shape
                    _, S, _ = keys.shape
                    H = attention_layer.n_heads

                    queries_proj = attention_layer.query_projection(queries)
                    keys_proj = attention_layer.key_projection(keys)
                    values_proj = attention_layer.value_projection(values)

                    queries_proj = queries_proj.view(B, L, H, -1)
                    keys_proj = keys_proj.view(B, S, H, -1)
                    values_proj = values_proj.view(B, S, H, -1)

                    out, attn = attention_layer.inner_attention(
                        queries_proj,
                        keys_proj,
                        values_proj,
                        attn_mask,
                        tau=tau,
                        delta=delta,
                    )

                    if self.current_mask is not None:
                        mask = self.current_mask.to(out.device)

                        # Global mask: [num_layers, num_heads]
                        if mask.ndim == 2:
                            layer_mask = mask[layer_idx].view(1, 1, H, 1)

                        # Batch mask: [original_batch, num_layers, num_heads]
                        elif mask.ndim == 3:
                            mask_batch = mask.shape[0]

                            # PatchTST encoder B can be original_batch * channel_count
                            if mask_batch != B:
                                if B % mask_batch != 0:
                                    raise ValueError(
                                        f"Cannot expand mask batch {mask_batch} to encoder batch {B}."
                                    )

                                repeat_factor = B // mask_batch
                                mask = mask.repeat_interleave(
                                    repeat_factor,
                                    dim=0,
                                )

                            layer_mask = mask[:, layer_idx, :].view(B, 1, H, 1)

                        else:
                            raise ValueError(
                                f"Unsupported mask shape: {mask.shape}"
                            )

                        out = out * layer_mask

                    out = out.view(B, L, -1)

                    return attention_layer.out_projection(out), attn

                return masked_forward

            attention_layer.forward = make_masked_forward(
                layer_idx,
                attention_layer,
            )

    def remove(self):
        for layer_idx, original_forward in self.original_forwards.items():
            self.model.encoder.attn_layers[layer_idx].attention.forward = original_forward

        self.original_forwards = {}
        self.current_mask = None

    def set_mask(self, mask):
        self.current_mask = mask.clone().float()

    def set_all_active(self):
        num_layers = len(self.model.encoder.attn_layers)
        num_heads = self.model.encoder.attn_layers[0].attention.n_heads

        self.current_mask = torch.ones(
            num_layers,
            num_heads,
            dtype=torch.float32,
        )

    def build_keep_mask_from_df(self, keep_df):
        num_layers = len(self.model.encoder.attn_layers)
        num_heads = self.model.encoder.attn_layers[0].attention.n_heads

        mask = torch.zeros(
            num_layers,
            num_heads,
            dtype=torch.float32,
        )

        for _, row in keep_df.iterrows():
            layer_idx = int(row["layer"])
            head_idx = int(row["head"])
            mask[layer_idx, head_idx] = 1.0

        return mask


In [17]:
mask_controller = DynamicHeadMaskController(
    model,
    num_channels=args.enc_in,
)

mask_controller.install()

num_layers = len(model.encoder.attn_layers)
num_heads = model.encoder.attn_layers[0].attention.n_heads

print("Layers:", num_layers)
print("Heads per layer:", num_heads)
print("Total heads:", num_layers * num_heads)

baseline_mask = torch.ones(num_layers, num_heads)

regime_masks = {
    regime: mask_controller.build_keep_mask_from_df(df_keep)
    for regime, df_keep in dynamic_keep_dfs.items()
}

for regime, mask in regime_masks.items():
    print(f"\n{regime} mask:")
    print(mask)
    print("Active heads:", int((mask == 1).sum().item()))
    print("Pruned heads:", int((mask == 0).sum().item()))

    assert int((mask == 1).sum().item()) == 18
    assert int((mask == 0).sum().item()) == 6


Layers: 3
Heads per layer: 8
Total heads: 24

trend mask:
tensor([[1., 1., 1., 1., 1., 0., 0., 1.],
        [1., 1., 1., 0., 1., 1., 1., 1.],
        [0., 1., 1., 0., 1., 1., 1., 0.]])
Active heads: 18
Pruned heads: 6

seasonal mask:
tensor([[1., 1., 1., 0., 0., 0., 1., 1.],
        [1., 0., 1., 1., 0., 1., 1., 1.],
        [0., 1., 1., 1., 1., 1., 1., 1.]])
Active heads: 18
Pruned heads: 6

residual mask:
tensor([[1., 1., 1., 1., 0., 1., 1., 1.],
        [1., 1., 0., 1., 0., 1., 1., 0.],
        [1., 1., 1., 1., 0., 1., 0., 1.]])
Active heads: 18
Pruned heads: 6


## 11. Batch dynamic mask oluşturma

In [18]:
def build_batch_dynamic_mask(
    window_ids,
    regime_df,
    regime_masks,
):
    batch_masks = []

    for window_id in window_ids:
        regime = regime_df.iloc[int(window_id)]["regime"]
        batch_masks.append(regime_masks[regime])

    batch_mask = torch.stack(batch_masks, dim=0)

    return batch_mask


## 12. Dynamic mask test

In [19]:
batch = next(iter(vali_loader))
batch_x, batch_y, batch_x_mark, batch_y_mark = batch

batch_x = batch_x.float().to(device)
batch_y = batch_y.float().to(device)
batch_x_mark = batch_x_mark.float().to(device)
batch_y_mark = batch_y_mark.float().to(device)

batch_size = batch_x.shape[0]
window_ids = list(range(batch_size))

batch_dynamic_mask = build_batch_dynamic_mask(
    window_ids=window_ids,
    regime_df=regime_df,
    regime_masks=regime_masks,
)

print("Batch dynamic mask shape:", batch_dynamic_mask.shape)

mask_controller.set_mask(baseline_mask)

with torch.no_grad():
    outputs_baseline = model(
        batch_x,
        batch_x_mark,
        batch_y,
        batch_y_mark,
    )

mask_controller.set_mask(batch_dynamic_mask)

with torch.no_grad():
    outputs_dynamic = model(
        batch_x,
        batch_x_mark,
        batch_y,
        batch_y_mark,
    )

diff = torch.mean(
    torch.abs(outputs_baseline - outputs_dynamic)
).item()

print("Baseline output shape:", outputs_baseline.shape)
print("Dynamic output shape:", outputs_dynamic.shape)
print("Mean absolute output difference:", diff)

assert diff > 0, "Dynamic mask output'u değiştirmedi. Masking çalışmıyor olabilir."

mask_controller.set_mask(baseline_mask)

Batch dynamic mask shape: torch.Size([32, 3, 8])
Baseline output shape: torch.Size([32, 96, 7])
Dynamic output shape: torch.Size([32, 96, 7])
Mean absolute output difference: 0.09279964864253998


## 13. Metric fonksiyonları

In [20]:
mse_criterion = nn.MSELoss(reduction="none")
mae_criterion = nn.L1Loss(reduction="none")

def compute_validation_baseline(
    model,
    loader,
    regime_df,
    mask_controller,
    baseline_mask,
    device,
    pred_len=96,
    desc="Baseline validation",
):
    model.eval()
    mask_controller.set_mask(baseline_mask)

    all_records = []
    global_index = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(
                batch_x,
                batch_x_mark,
                batch_y,
                batch_y_mark,
            )

            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            batch_size = batch_x.shape[0]

            for i in range(batch_size):
                window_id = global_index + i
                regime = regime_df.iloc[window_id]["regime"]

                all_records.append({
                    "window_id": window_id,
                    "regime": regime,
                    "mse": float(mse_per_sample[i].detach().cpu()),
                    "mae": float(mae_per_sample[i].detach().cpu()),
                })

            global_index += batch_size

    result_df = pd.DataFrame(all_records)

    regime_summary = (
        result_df
        .groupby("regime")
        .agg(
            mse=("mse", "mean"),
            mae=("mae", "mean"),
            count=("window_id", "count"),
        )
        .reset_index()
    )

    overall = {
        "overall_mse": float(result_df["mse"].mean()),
        "overall_mae": float(result_df["mae"].mean()),
    }

    return {
        "overall": overall,
        "regime_summary": regime_summary,
        "window_losses": result_df,
    }


def compute_validation_dynamic(
    model,
    loader,
    regime_df,
    regime_masks,
    mask_controller,
    device,
    pred_len=96,
    desc="Dynamic regime-aware validation",
):
    model.eval()

    all_records = []
    global_index = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            batch_size = batch_x.shape[0]
            window_ids = list(range(global_index, global_index + batch_size))

            batch_dynamic_mask = build_batch_dynamic_mask(
                window_ids=window_ids,
                regime_df=regime_df,
                regime_masks=regime_masks,
            )

            mask_controller.set_mask(batch_dynamic_mask)

            outputs = model(
                batch_x,
                batch_x_mark,
                batch_y,
                batch_y_mark,
            )

            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            for i in range(batch_size):
                window_id = global_index + i
                regime = regime_df.iloc[window_id]["regime"]

                all_records.append({
                    "window_id": window_id,
                    "regime": regime,
                    "mse": float(mse_per_sample[i].detach().cpu()),
                    "mae": float(mae_per_sample[i].detach().cpu()),
                })

            global_index += batch_size

    result_df = pd.DataFrame(all_records)

    regime_summary = (
        result_df
        .groupby("regime")
        .agg(
            mse=("mse", "mean"),
            mae=("mae", "mean"),
            count=("window_id", "count"),
        )
        .reset_index()
    )

    overall = {
        "overall_mse": float(result_df["mse"].mean()),
        "overall_mae": float(result_df["mae"].mean()),
    }

    return {
        "overall": overall,
        "regime_summary": regime_summary,
        "window_losses": result_df,
    }


## 14. Baseline ve dynamic 75% validation hesapla

In [21]:
baseline_val = compute_validation_baseline(
    model=model,
    loader=vali_loader,
    regime_df=regime_df,
    mask_controller=mask_controller,
    baseline_mask=baseline_mask,
    device=device,
    pred_len=args.pred_len,
    desc="Baseline validation",
)

dynamic_75_val = compute_validation_dynamic(
    model=model,
    loader=vali_loader,
    regime_df=regime_df,
    regime_masks=regime_masks,
    mask_controller=mask_controller,
    device=device,
    pred_len=args.pred_len,
    desc="Dynamic 75% keep validation",
)

print("Baseline validation overall:")
print(baseline_val["overall"])

print("\nDynamic 75% validation overall:")
print(dynamic_75_val["overall"])

print("\nBaseline regime summary:")
display(baseline_val["regime_summary"])

print("\nDynamic 75% regime summary:")
display(dynamic_75_val["regime_summary"])

Baseline validation:   0%|          | 0/88 [00:00<?, ?it/s]

Dynamic 75% keep validation:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline validation overall:
{'overall_mse': 0.6780664091023552, 'overall_mae': 0.5550830315858715}

Dynamic 75% validation overall:
{'overall_mse': 0.6596687611867966, 'overall_mae': 0.5511876718039145}

Baseline regime summary:


,regime,mse,mae,count
0,residual,0.645579,0.545933,359
1,seasonal,0.672409,0.558120,292
2,trend,0.684306,0.556207,2134



Dynamic 75% regime summary:


,regime,mse,mae,count
0,residual,0.660827,0.551801,359
1,seasonal,0.676286,0.559426,292
2,trend,0.657200,0.549957,2134


## 15. Dynamic 75% overall comparison

In [22]:
baseline_val_overall = baseline_val["overall"]
dynamic_75_overall = dynamic_75_val["overall"]

dynamic_75_comparison = pd.DataFrame([
    {
        "setting": "B4_no_pruning",
        "selection_type": "none",
        "active_heads_per_sample": 24,
        "pruned_heads_per_sample": 0,
        "selection_ratio_active": 1.0,
        "validation_mse": baseline_val_overall["overall_mse"],
        "validation_mae": baseline_val_overall["overall_mae"],
    },
    {
        "setting": "B4_dynamic_regime_aware_75_keep",
        "selection_type": "dynamic_regime_aware",
        "active_heads_per_sample": 18,
        "pruned_heads_per_sample": 6,
        "selection_ratio_active": 0.75,
        "validation_mse": dynamic_75_overall["overall_mse"],
        "validation_mae": dynamic_75_overall["overall_mae"],
    },
])

baseline_mse = dynamic_75_comparison.loc[
    dynamic_75_comparison["setting"] == "B4_no_pruning",
    "validation_mse",
].iloc[0]

baseline_mae = dynamic_75_comparison.loc[
    dynamic_75_comparison["setting"] == "B4_no_pruning",
    "validation_mae",
].iloc[0]

dynamic_75_comparison["delta_validation_mse"] = (
    dynamic_75_comparison["validation_mse"] - baseline_mse
)

dynamic_75_comparison["delta_validation_mae"] = (
    dynamic_75_comparison["validation_mae"] - baseline_mae
)

dynamic_75_comparison["relative_mse_change_percent"] = (
    dynamic_75_comparison["delta_validation_mse"]
    / baseline_mse
    * 100
)

dynamic_75_comparison["relative_mae_change_percent"] = (
    dynamic_75_comparison["delta_validation_mae"]
    / baseline_mae
    * 100
)

display(dynamic_75_comparison)

,setting,selection_type,active_heads_per_sample,pruned_heads_per_sample,selection_ratio_active,validation_mse,validation_mae,delta_validation_mse,delta_validation_mae,relative_mse_change_percent,relative_mae_change_percent
0,B4_no_pruning,none,24,0,1.00,0.678066,0.555083,0.000000,0.000000,0.000000,0.000000
1,B4_dynamic_regime_aware_75_keep,dynamic_regime_aware,18,6,0.75,0.659669,0.551188,-0.018398,-0.003895,-2.713252,-0.701762


## 16. Regime-level dynamic 75% delta

In [23]:
baseline_regime = baseline_val["regime_summary"].copy()
baseline_regime["setting"] = "B4_no_pruning"

dynamic_75_regime = dynamic_75_val["regime_summary"].copy()
dynamic_75_regime["setting"] = "B4_dynamic_regime_aware_75_keep"

dynamic_75_regime_comparison = pd.concat(
    [baseline_regime, dynamic_75_regime],
    ignore_index=True,
)

display(dynamic_75_regime_comparison)

baseline_regime_ref = baseline_regime[
    ["regime", "mse", "mae"]
].rename(
    columns={
        "mse": "baseline_mse",
        "mae": "baseline_mae",
    }
)

dynamic_75_regime_delta = dynamic_75_regime.merge(
    baseline_regime_ref,
    on="regime",
    how="left",
)

dynamic_75_regime_delta["delta_mse"] = (
    dynamic_75_regime_delta["mse"]
    - dynamic_75_regime_delta["baseline_mse"]
)

dynamic_75_regime_delta["delta_mae"] = (
    dynamic_75_regime_delta["mae"]
    - dynamic_75_regime_delta["baseline_mae"]
)

dynamic_75_regime_delta["relative_mse_change_percent"] = (
    dynamic_75_regime_delta["delta_mse"]
    / dynamic_75_regime_delta["baseline_mse"]
    * 100
)

dynamic_75_regime_delta["relative_mae_change_percent"] = (
    dynamic_75_regime_delta["delta_mae"]
    / dynamic_75_regime_delta["baseline_mae"]
    * 100
)

display(dynamic_75_regime_delta)

,regime,mse,mae,count,setting
0,residual,0.645579,0.545933,359,B4_no_pruning
1,seasonal,0.672409,0.558120,292,B4_no_pruning
2,trend,0.684306,0.556207,2134,B4_no_pruning
3,residual,0.660827,0.551801,359,B4_dynamic_regime_aware_75_keep
4,seasonal,0.676286,0.559426,292,B4_dynamic_regime_aware_75_keep
5,trend,0.657200,0.549957,2134,B4_dynamic_regime_aware_75_keep


,regime,mse,mae,count,setting,baseline_mse,baseline_mae,delta_mse,delta_mae,relative_mse_change_percent,relative_mae_change_percent
0,residual,0.660827,0.551801,359,B4_dynamic_regime_aware_75_keep,0.645579,0.545933,0.015248,0.005868,2.361873,1.074883
1,seasonal,0.676286,0.559426,292,B4_dynamic_regime_aware_75_keep,0.672409,0.558120,0.003877,0.001306,0.576649,0.234036
2,trend,0.657200,0.549957,2134,B4_dynamic_regime_aware_75_keep,0.684306,0.556207,-0.027106,-0.006250,-3.961053,-1.123611


## 17. Static 25% ve Dynamic 50% sonuçlarını varsa oku

In [24]:
static_val_path = STATIC_DIR / "validation_overall_comparison.csv"
dynamic_50_val_path = DYNAMIC_50_DIR / "dynamic_validation_overall_comparison.csv"

static_available = static_val_path.exists()
dynamic_50_available = dynamic_50_val_path.exists()

print("Static 25% available:", static_available)
print("Dynamic 50% available:", dynamic_50_available)

if static_available:
    static_val_comparison = pd.read_csv(static_val_path)
    display(static_val_comparison)

if dynamic_50_available:
    dynamic_50_comparison = pd.read_csv(dynamic_50_val_path)
    display(dynamic_50_comparison)


Static 25% available: True
Dynamic 50% available: True


,setting,pruning_type,pruned_heads,active_heads,pruning_ratio,validation_mse,validation_mae,delta_validation_mse,delta_validation_mae,relative_mse_change_percent,relative_mae_change_percent
0,B4_no_pruning,none,0,24,0.00,0.678066,0.555083,0.000000,0.000000,0.000000,0.000000
1,B4_static_pruning_25,static_overall_importance,6,18,0.25,0.653658,0.550686,-0.024409,-0.004397,-3.599745,-0.792198


,setting,selection_type,active_heads_per_sample,pruned_heads_per_sample,selection_ratio_active,validation_mse,validation_mae,delta_validation_mse,delta_validation_mae,relative_mse_change_percent,relative_mae_change_percent
0,B4_no_pruning,none,24,0,1.0,0.678066,0.555083,0.000000,0.000000,0.000000,0.000000
1,B4_dynamic_regime_aware_50_keep,dynamic_regime_aware,12,12,0.5,0.667840,0.555278,-0.010226,0.000195,-1.508114,0.035153


## 18. Combined validation comparison

In [25]:
combined_rows = []

# Baseline row from current run
combined_rows.append({
    "setting": "B4_no_pruning",
    "method": "none",
    "active_heads_per_sample": 24,
    "pruned_heads_per_sample": 0,
    "validation_mse": baseline_val_overall["overall_mse"],
    "validation_mae": baseline_val_overall["overall_mae"],
})

# Static rows if available
if static_available:
    static_pruned_row = static_val_comparison[
        static_val_comparison["setting"] == "B4_static_pruning_25"
    ].iloc[0]

    combined_rows.append({
        "setting": "B4_static_pruning_25",
        "method": "static_overall_importance",
        "active_heads_per_sample": int(static_pruned_row["active_heads"]),
        "pruned_heads_per_sample": int(static_pruned_row["pruned_heads"]),
        "validation_mse": float(static_pruned_row["validation_mse"]),
        "validation_mae": float(static_pruned_row["validation_mae"]),
    })

# Dynamic 50 if available
if dynamic_50_available:
    dynamic_50_row = dynamic_50_comparison[
        dynamic_50_comparison["setting"] == "B4_dynamic_regime_aware_50_keep"
    ].iloc[0]

    combined_rows.append({
        "setting": "B4_dynamic_regime_aware_50_keep",
        "method": "dynamic_regime_aware",
        "active_heads_per_sample": int(dynamic_50_row["active_heads_per_sample"]),
        "pruned_heads_per_sample": int(dynamic_50_row["pruned_heads_per_sample"]),
        "validation_mse": float(dynamic_50_row["validation_mse"]),
        "validation_mae": float(dynamic_50_row["validation_mae"]),
    })

# Dynamic 75 current
combined_rows.append({
    "setting": "B4_dynamic_regime_aware_75_keep",
    "method": "dynamic_regime_aware",
    "active_heads_per_sample": 18,
    "pruned_heads_per_sample": 6,
    "validation_mse": dynamic_75_overall["overall_mse"],
    "validation_mae": dynamic_75_overall["overall_mae"],
})

combined_val_comparison = pd.DataFrame(combined_rows)

baseline_combined_mse = combined_val_comparison.loc[
    combined_val_comparison["setting"] == "B4_no_pruning",
    "validation_mse",
].iloc[0]

baseline_combined_mae = combined_val_comparison.loc[
    combined_val_comparison["setting"] == "B4_no_pruning",
    "validation_mae",
].iloc[0]

combined_val_comparison["delta_validation_mse"] = (
    combined_val_comparison["validation_mse"]
    - baseline_combined_mse
)

combined_val_comparison["delta_validation_mae"] = (
    combined_val_comparison["validation_mae"]
    - baseline_combined_mae
)

combined_val_comparison["relative_mse_change_percent"] = (
    combined_val_comparison["delta_validation_mse"]
    / baseline_combined_mse
    * 100
)

combined_val_comparison["relative_mae_change_percent"] = (
    combined_val_comparison["delta_validation_mae"]
    / baseline_combined_mae
    * 100
)

display(combined_val_comparison.sort_values("validation_mse"))

,setting,method,active_heads_per_sample,pruned_heads_per_sample,validation_mse,validation_mae,delta_validation_mse,delta_validation_mae,relative_mse_change_percent,relative_mae_change_percent
1,B4_static_pruning_25,static_overall_importance,18,6,0.653658,0.550686,-0.024409,-0.004397,-3.599745,-0.792198
3,B4_dynamic_regime_aware_75_keep,dynamic_regime_aware,18,6,0.659669,0.551188,-0.018398,-0.003895,-2.713252,-0.701762
2,B4_dynamic_regime_aware_50_keep,dynamic_regime_aware,12,12,0.667840,0.555278,-0.010226,0.000195,-1.508114,0.035153
0,B4_no_pruning,none,24,0,0.678066,0.555083,0.000000,0.000000,0.000000,0.000000


## 19. Sonuçları kaydet

In [26]:
dynamic_75_comparison.to_csv(
    DYNAMIC_75_DIR / "dynamic_75_validation_overall_comparison.csv",
    index=False,
)

dynamic_75_regime_comparison.to_csv(
    DYNAMIC_75_DIR / "dynamic_75_validation_regime_comparison.csv",
    index=False,
)

dynamic_75_regime_delta.to_csv(
    DYNAMIC_75_DIR / "dynamic_75_validation_regime_delta.csv",
    index=False,
)

dynamic_75_val["window_losses"].to_csv(
    DYNAMIC_75_DIR / "dynamic_75_validation_window_losses.csv",
    index=False,
)

baseline_val["window_losses"].to_csv(
    DYNAMIC_75_DIR / "baseline_validation_window_losses.csv",
    index=False,
)

combined_val_comparison.to_csv(
    DYNAMIC_75_DIR / "combined_validation_comparison.csv",
    index=False,
)

# Maskeleri kaydet
for regime, mask in regime_masks.items():
    mask_df = pd.DataFrame(
        mask.numpy(),
        index=[f"layer_{i}" for i in range(mask.shape[0])],
        columns=[f"head_{j}" for j in range(mask.shape[1])],
    )
    mask_df.to_csv(
        DYNAMIC_75_DIR / f"{regime}_dynamic_keep_75_mask.csv"
    )

# Keep head listeleri
for regime, keep_df in dynamic_keep_dfs.items():
    keep_df.to_csv(
        DYNAMIC_75_DIR / f"{regime}_dynamic_keep_75_heads.csv",
        index=False,
    )

print("Saved dynamic 75% experiment outputs to:")
print(DYNAMIC_75_DIR)

print("\nFiles:")
for path in sorted(DYNAMIC_75_DIR.iterdir()):
    print(path.name)

Saved dynamic 75% experiment outputs to:
/content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_dynamic_regime_aware_75_keep

Files:
baseline_validation_window_losses.csv
combined_validation_comparison.csv
dynamic_75_validation_overall_comparison.csv
dynamic_75_validation_regime_comparison.csv
dynamic_75_validation_regime_delta.csv
dynamic_75_validation_window_losses.csv
residual_dynamic_keep_75_heads.csv
residual_dynamic_keep_75_mask.csv
seasonal_dynamic_keep_75_heads.csv
seasonal_dynamic_keep_75_mask.csv
trend_dynamic_keep_75_heads.csv
trend_dynamic_keep_75_mask.csv


## 20. Kısa yorum üret

In [27]:
dynamic_75_row = dynamic_75_comparison[
    dynamic_75_comparison["setting"] == "B4_dynamic_regime_aware_75_keep"
].iloc[0]

print("Dynamic 75% validation:")
print(
    f"Dynamic regime-aware 75% keep changed validation MSE by "
    f"{dynamic_75_row['delta_validation_mse']:.6f} "
    f"({dynamic_75_row['relative_mse_change_percent']:.3f}%)."
)

print(
    f"Dynamic regime-aware 75% keep changed validation MAE by "
    f"{dynamic_75_row['delta_validation_mae']:.6f} "
    f"({dynamic_75_row['relative_mae_change_percent']:.3f}%)."
)

print("\nRegime-level dynamic 75% validation delta:")
display(
    dynamic_75_regime_delta[
        [
            "regime",
            "baseline_mse",
            "mse",
            "delta_mse",
            "relative_mse_change_percent",
            "baseline_mae",
            "mae",
            "delta_mae",
            "relative_mae_change_percent",
        ]
    ]
)

print("\nCombined validation comparison:")
display(combined_val_comparison.sort_values("validation_mse"))


Dynamic 75% validation:
Dynamic regime-aware 75% keep changed validation MSE by -0.018398 (-2.713%).
Dynamic regime-aware 75% keep changed validation MAE by -0.003895 (-0.702%).

Regime-level dynamic 75% validation delta:


,regime,baseline_mse,mse,delta_mse,relative_mse_change_percent,baseline_mae,mae,delta_mae,relative_mae_change_percent
0,residual,0.645579,0.660827,0.015248,2.361873,0.545933,0.551801,0.005868,1.074883
1,seasonal,0.672409,0.676286,0.003877,0.576649,0.558120,0.559426,0.001306,0.234036
2,trend,0.684306,0.657200,-0.027106,-3.961053,0.556207,0.549957,-0.006250,-1.123611



Combined validation comparison:


,setting,method,active_heads_per_sample,pruned_heads_per_sample,validation_mse,validation_mae,delta_validation_mse,delta_validation_mae,relative_mse_change_percent,relative_mae_change_percent
1,B4_static_pruning_25,static_overall_importance,18,6,0.653658,0.550686,-0.024409,-0.004397,-3.599745,-0.792198
3,B4_dynamic_regime_aware_75_keep,dynamic_regime_aware,18,6,0.659669,0.551188,-0.018398,-0.003895,-2.713252,-0.701762
2,B4_dynamic_regime_aware_50_keep,dynamic_regime_aware,12,12,0.667840,0.555278,-0.010226,0.000195,-1.508114,0.035153
0,B4_no_pruning,none,24,0,0.678066,0.555083,0.000000,0.000000,0.000000,0.000000
